# CSP Scanner — Stepwise Test Notebook

Ziel: jede Schicht des Scanners einzeln ausprobieren, Zwischenergebnisse inspizieren und die IB-Connection effizient nutzen.

**Voraussetzungen**
- TWS oder IB Gateway läuft (Paper-Account, Port 7497)
- `pip install -r ../requirements.txt` ist erledigt
- Optional: `pip install jupyterlab pandas`

**Empfohlene Reihenfolge**
1. Imports & Jupyter-Loop starten
2. Config & Watchlist laden
3. IB-Verbindung aufbauen
4. Eine Aktie qualifizieren + Spot holen
5. Option-Chain-Parameter (Expiries, Strikes)
6. DTE-Filter anwenden
7. Put-Quotes für eine Expiry holen
8. Liquiditätsfilter + Yield-Ranking manuell
9. Selector pro Ticker (komplett)
10. T-Bill-Matching
11. Kompletten Lauf über Watchlist + Excel-Report
12. Aufräumen (Disconnect)


## Schritt 1 — Imports & Jupyter-Loop

`ib_async` nutzt asyncio. In Jupyter muss der laufende Event-Loop "gepatched" werden, sonst blockieren `ib.connect()` und `ib.sleep()`.


In [1]:
import sys
from pathlib import Path
from datetime import datetime, timedelta

# Projekt-Root auf den Python-Pfad setzen, damit `from src...` funktioniert.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

import pandas as pd
from ib_async import util

# Jupyter-kompatibler Event-Loop für ib_async:
util.startLoop()
print('Jupyter event loop ready.')


Project root: /Users/stefan_pro/Documents/Claude/Projects/csp_scanner
Jupyter event loop ready.


## Schritt 2 — Config & Watchlist laden

Keine IB-Calls nötig. Reines YAML-Parsing. Wenn das fehlschlägt, ist `config/*.yaml` defekt.


In [2]:
from src.watchlist import load_settings, load_watchlist

SETTINGS_PATH  = PROJECT_ROOT / 'config' / 'settings_all.yaml'
WATCHLIST_PATH = PROJECT_ROOT / 'config' / 'watchlist_sap.yaml'

settings  = load_settings(SETTINGS_PATH)
watchlist = load_watchlist(WATCHLIST_PATH)

print(f'{len(watchlist)} watchlist entries loaded:')
for e in watchlist:
    print(f'  {e.symbol:<6} max_strike={e.max_strike:<8.2f} max_contracts={e.max_contracts}')
print()
print('Options config:', settings.options)
print('IB config     :', settings.ib)
print(settings)
print(watchlist)


1 watchlist entries loaded:
  SAP    max_strike=140.00   max_contracts=10

Options config: OptionsConfig(dte_min=60, dte_max=720, expiry_filter='monthlies', max_spread_pct=100.0, require_positive_bid=True, min_annualized_yield=0.0, top_n_per_ticker=25)
IB config     : IBConfig(host='127.0.0.1', port=4001, client_id=1, market_data_type=2, request_timeout_s=15)
Settings(ib=IBConfig(host='127.0.0.1', port=4001, client_id=1, market_data_type=2, request_timeout_s=15), options=OptionsConfig(dte_min=60, dte_max=720, expiry_filter='monthlies', max_spread_pct=100.0, require_positive_bid=True, min_annualized_yield=0.0, top_n_per_ticker=25), tbill=TBillConfig(enabled=True, buckets_days=[28, 91, 182, 364], fallback_yield=0.04), report=ReportConfig(output_dir='output', filename_prefix='csp_scan', open_after_run=False), store=StoreConfig(enabled=True, db_path='data/csp_history.duckdb'))
[WatchlistEntry(symbol='SAP', stk_exchange='SMART', opt_exchange='SMART', trading_class='SAP', currency='EUR', max

## Schritt 3 — IB-Verbindung aufbauen

Baut die Verbindung zu TWS/Gateway auf. `market_data_type=3` (delayed) läuft ohne Market-Data-Subscription und ist zum Testen vorzuziehen.


In [3]:
from src.ib_client import IBClient

client = IBClient(
    host=settings.ib.host,
    port=settings.ib.port,
    client_id=settings.ib.client_id,
    market_data_type=settings.ib.market_data_type,
    request_timeout_s=settings.ib.request_timeout_s,
)
client.connect()
print('Connected:', client.ib.isConnected())
print('Server version:', client.ib.client.serverVersion() if client.ib.isConnected() else 'n/a')


Connected: True
Server version: 178


## Schritt 4 — Aktie qualifizieren + Spot-Preis

Erster IB-Roundtrip. Wenn das nicht funktioniert: Port falsch, TWS nicht gestartet, oder Symbol ungültig.


In [4]:
# Einen Ticker zum Testen auswählen — ersten Eintrag der Watchlist.
test_entry = watchlist[0]
print(f'Testing with {test_entry.symbol}  (max_strike={test_entry.max_strike})')

stock = client.qualify_stock(test_entry.symbol, test_entry.stk_exchange, test_entry.currency)
print('Qualified:', stock)

spot = client.spot_price(stock)
print(f'Spot price: {spot:.2f}')




Testing with SAP  (max_strike=140.0)
Qualified: Stock(conId=14204, symbol='SAP', exchange='SMART', primaryExchange='IBIS', currency='EUR', localSymbol='SAP', tradingClass='XETRA')
Spot price: 147.06


## Schritt 5 — Option-Chain-Parameter

`reqSecDefOptParams` liefert **alle** Expiries und Strikes, die IB für das Underlying listet. Das wird hier noch nicht pro Kontrakt gequotet — nur die Struktur der Chain.


In [ ]:
chain = client.option_chain_params(stock)
# Override with watchlist values if configured
opt_exchange = test_entry.opt_exchange if test_entry.opt_exchange != "SMART" else chain["exchange"]
trading_class = test_entry.trading_class if test_entry.trading_class else chain["trading_class"]
print(f"Exchange: {opt_exchange}  TradingClass: {trading_class}")
print(f"Expiries ({len(chain['expiries'])}): first 10 -> {chain['expiries'][:10]}")
print(f"Strikes ({len(chain['strikes'])}):   range [{min(chain['strikes']):.2f}, {max(chain['strikes']):.2f}]")


Exchange: FTA  TradingClass: APQ
Expiries (10): first 10 -> ['20260515', '20260619', '20260717', '20260918', '20261218', '20270319', '20270618', '20271217', '20280616', '20281215']
Strikes (75):   range [60.00, 500.00]


## Schritt 6 — DTE-Filter auf Expiries

Aus den gelisteten Expiries die innerhalb des konfigurierten DTE-Fensters auswählen.


In [9]:
from src.ib_client import dte_from_expiry

today = datetime.utcnow()
expiries_df = pd.DataFrame({
    'expiry': chain['expiries'],
    'dte':    [dte_from_expiry(e, today) for e in chain['expiries']],
})
valid = expiries_df.query(
    f'dte >= {settings.options.dte_min} and dte <= {settings.options.dte_max}'
).reset_index(drop=True)
print(f'Valid expiries within [{settings.options.dte_min}, {settings.options.dte_max}] DTE:')
valid


Valid expiries within [60, 720] DTE:


/var/folders/3d/78d9j10d6rz512qz8ckx18h80000gn/T/ipykernel_77785/3099522143.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


,expiry,dte
0,20260717,80
1,20260918,143
2,20261218,234
3,20270319,325
4,20270618,416
5,20271217,598


## Schritt 7 — Put-Quotes für eine einzelne Expiry

Hier wird es "teuer": für jeden Strike wird ein Option-Kontrakt qualifiziert + ein Market-Data-Request abgesetzt. Mit `market_data_type=3` (delayed) kommen Preise auch ohne Live-Abo an, allerdings können Greeks und Open Interest fehlen.


In [ ]:
test_expiry = valid.iloc[0]['expiry']
print(f'Fetching puts for {test_entry.symbol} expiry {test_expiry}')

# Strikes innerhalb sinnvoller Bandbreite: <= max_strike und nicht unter 50% des Spots
lower = max(spot * 0.5, 0)
cand_strikes = [k for k in chain['strikes'] if lower <= k <= test_entry.max_strike]
print(f'Considering {len(cand_strikes)} strikes in [{lower:.2f}, {test_entry.max_strike:.2f}]')

quotes = client.fetch_put_quotes(
    symbol=test_entry.symbol,
    expiry=test_expiry,
    strikes=cand_strikes,
    exchange='EUREX',
    #trading_class=chain['trading_class'],
    trading_class='SAP',
    currency=test_entry.currency,
)
print(f'Received {len(quotes)} quotes')

df_quotes = pd.DataFrame([q.__dict__ for q in quotes])
df_quotes['spread_pct'] = [q.spread_pct for q in quotes]
df_quotes.head(20)


## Schritt 8 — Manuelle Filterung + Yield-Ranking

Nur zur Veranschaulichung: wir wenden dieselben Filter wie der Selector manuell an, damit wir die Zwischenergebnisse sehen.


In [ ]:
from src.option_selector import CSPCandidate

filtered = []
for q in quotes:
    # Filter für Testzwecke deaktiviert
    # Underlying stempeln, falls Greeks es nicht mitliefern
    #if not q.underlying_price or q.underlying_price != q.underlying_price:
    #    q.underlying_price = spot
    cand = CSPCandidate.from_quote(q, today)
    #if cand is None:
        #continue
    filtered.append(cand)

filtered.sort(key=lambda c: c.annualized_yield, reverse=True)
df_top = pd.DataFrame([c.__dict__ for c in filtered])
print(f'{len(filtered)} candidates after filters')
if not df_top.empty:
    df_top[['symbol','expiry','dte','strike','mid','bid','ask','spread_pct','annualized_yield','delta','iv','open_interest']]


## Schritt 9 — Selector komplett auf einem Ticker

Die gesamte Selector-Logik (über alle Expiries) aus Modul `option_selector.py`.


In [ ]:
from src.option_selector import CSPSelector

selector = CSPSelector(client, settings.options)
candidates = selector.scan_ticker(test_entry, today=today)
print(f'{len(candidates)} total candidates for {test_entry.symbol}')

df_cand = pd.DataFrame([c.__dict__ for c in candidates])
if not df_cand.empty:
    df_cand[['symbol','expiry','dte','strike','mid','spread_pct','annualized_yield','delta','open_interest']].head(20)


## Schritt 10 — T-Bill-Matching

Für eine DTE den passenden T-Bill-Bucket holen und die erwartete Zinszahlung auf den zu parkenden Cash berechnen.


In [ ]:
from src.tbill import TBillMatcher

matcher = TBillMatcher(client, settings.tbill)

if candidates:
    c = candidates[0]
    m = matcher.match(c.dte, cash_usd=c.cash_required)
    print(f'Top candidate: {c.symbol} K={c.strike} DTE={c.dte}')
    print(f'  cash required     : {c.cash_required:,.0f} USD')
    print(f'  bucket chosen     : {m.bucket_days} days')
    print(f'  T-Bill yield      : {m.yield_pct*100:.2f}%  (source={m.source})')
    print(f'  projected interest: {m.projected_interest:,.2f} USD')
    print(f'  premium           : {c.premium:,.2f} USD')
    print(f'  total if OTM      : {c.premium + m.projected_interest:,.2f} USD')
else:
    print('No candidates to match T-Bill against.')


## Schritt 11 — Kompletter Lauf + Excel-Report

Jetzt alles zusammen: für die ganze Watchlist scannen, T-Bill-Matches bauen, Excel schreiben.


In [ ]:
from src.report import write_report

candidates_by_ticker = {}
tbill_matches = {}

for entry in watchlist:
    try:
        cs = selector.scan_ticker(entry, today=today)
    except Exception as e:
        print(f'[WARN] scan failed for {entry.symbol}: {e}')
        cs = []
    candidates_by_ticker[entry.symbol] = cs

    per_expiry = {}
    for c in cs:
        if c.expiry not in per_expiry:
            per_expiry[c.expiry] = matcher.match(c.dte, cash_usd=c.cash_required)
    tbill_matches[entry.symbol] = per_expiry

total = sum(len(v) for v in candidates_by_ticker.values())
print(f'Total candidates across watchlist: {total}')

stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
out = PROJECT_ROOT / 'output' / f'notebook_{stamp}.xlsx'
path = write_report(
    output_path=out,
    candidates_by_ticker=candidates_by_ticker,
    tbill_matches=tbill_matches,
    matcher=matcher,
    watchlist=watchlist,
    settings=settings,
    run_ts=datetime.utcnow(),
)
print(f'Report written to: {path}')


## Schritt 12 — Connection schließen

Sauber aufräumen, damit die IB-Client-ID nicht blockiert bleibt.


In [ ]:
client.disconnect()
print('Disconnected:', not client.ib.isConnected())


---

## Troubleshooting

| Symptom                                      | Ursache / Abhilfe                                                                 |
|----------------------------------------------|----------------------------------------------------------------------------------|
| `TimeoutError` bei `connect()`               | TWS/Gateway läuft nicht, Port falsch (7497 Paper / 7496 Live), Firewall          |
| `clientId already in use`                    | `ib.client_id` in `settings.yaml` ändern                                         |
| `qualifyContracts` gibt leere Liste zurück   | Symbol falsch oder nicht auf SMART; `stock.primaryExchange` setzen               |
| Spot-Preis ist NaN                           | Keine Market-Data-Subscription → `market_data_type: 3` nutzen oder länger warten |
| Leere Option-Chain                           | `secType`/`tradingClass` passt nicht; bei Weeklies alternative TradingClass      |
| Greeks fehlen (Delta/IV=NaN)                 | Delayed Data liefert oft keine Greeks; `market_data_type: 1` + Subscription      |
| Viele Requests hintereinander = Pacing-Error | In `fetch_put_quotes` längeres `ib.sleep(...)` oder kleinere Batches             |
